In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer import AerSimulator

print("Qiskit version:", qiskit.__version__)
sim = AerSimulator()

## 1. Defining single-qubit states
A qubit state is $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ with $|\alpha|^2 + |\beta|^2 = 1$.

In [ ]:
zero = Statevector.from_label('0')
one  = Statevector.from_label('1')
plus = Statevector.from_label('+')
minus = Statevector.from_label('-')
plus_i = Statevector.from_label('r')   # (|0> + i|1>)/sqrt(2)
minus_i = Statevector.from_label('l')  # (|0> - i|1>)/sqrt(2)

for name, sv in [('|0>', zero), ('|1>', one), ('|+>', plus), ('|->', minus), ('|+i>', plus_i), ('|-i>', minus_i)]:
    print(f"{name:5s} amplitudes = {np.round(sv.data, 4)}  probs = {np.round(sv.probabilities(), 4)}")

# Custom state from amplitudes
alpha, beta = 1/np.sqrt(3), np.sqrt(2/3) * np.exp(1j*np.pi/4)
custom = Statevector([alpha, beta])
assert np.isclose(np.sum(np.abs(custom.data)**2), 1.0)
assert custom.is_valid()
print("Custom state valid and normalized:", custom.is_valid())

## 2. Superposition and measurement statistics
Build $H|0\rangle$ in a circuit, measure 4096 shots, and compare with the theoretical 50/50 split.

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)
print(qc.draw())

shots = 4096
counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
print(counts)
plot_histogram(counts)
plt.show()

p0 = counts.get('0', 0) / shots
assert abs(p0 - 0.5) < 0.05, "H|0> should give ~50/50"

In [ ]:
# Unequal superposition: Ry(theta)|0> gives P(1) = sin^2(theta/2)
def measure_ry(theta, shots=8192):
    qc = QuantumCircuit(1, 1)
    qc.ry(theta, 0)
    qc.measure(0, 0)
    c = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    return c.get('1', 0) / shots

thetas = np.linspace(0, np.pi, 9)
empirical = [measure_ry(t) for t in thetas]
theory = np.sin(thetas/2)**2

plt.plot(thetas, theory, label='theory sin²(θ/2)')
plt.scatter(thetas, empirical, color='red', label='simulated')
plt.xlabel('θ'); plt.ylabel('P(1)'); plt.legend(); plt.show()
assert np.allclose(empirical, theory, atol=0.03)

## 3. Global phase vs. relative phase
Global phase (multiplying the whole state by $e^{i\gamma}$) is unobservable. Relative phase changes the state (e.g. $|+\rangle$ vs. $|-\rangle$) and shows up after a Hadamard.

In [ ]:
# Global phase: |+> and e^{i*0.7}|+> are indistinguishable
psi = plus
psi_global = Statevector(np.exp(1j*0.7) * plus.data)
print("Equivalent up to global phase (states equal?):", psi.equiv(psi_global))
print("Raw amplitudes differ:", not np.allclose(psi.data, psi_global.data))
assert psi.equiv(psi_global)
assert np.allclose(psi.probabilities(), psi_global.probabilities())

# Relative phase: |+> vs |-> give identical Z-basis probabilities but are different states
print("P(|+>) =", plus.probabilities(), " P(|->) =", minus.probabilities())
assert not plus.equiv(minus)

# Reveal the relative phase by applying H before measuring
for label, sv in [('|+>', plus), ('|->', minus)]:
    after_h = sv.evolve(Operator.from_label('H'))
    print(f"H{label} probabilities:", np.round(after_h.probabilities(), 4))
assert np.allclose(plus.evolve(Operator.from_label('H')).probabilities(), [1, 0])
assert np.allclose(minus.evolve(Operator.from_label('H')).probabilities(), [0, 1])

## 4. Single-qubit gates: X, Y, Z, H, S, T
Print each gate's matrix, apply it to $|0\rangle$ and $|+\rangle$, and confirm unitarity.

In [ ]:
gates = {
    'X': Operator.from_label('X'),
    'Y': Operator.from_label('Y'),
    'Z': Operator.from_label('Z'),
    'H': Operator.from_label('H'),
    'S': Operator.from_label('S'),
    'T': Operator.from_label('T'),
}
np.set_printoptions(precision=4, suppress=True)
for name, op in gates.items():
    U = op.data
    print(f"--- {name} ---")
    print(U)
    assert np.allclose(U.conj().T @ U, np.eye(2)), f"{name} not unitary"
print("All gates unitary.")

In [ ]:
# Action of each gate on |0> and |+>
print(f"{'gate':<5}{'on |0>':<28}{'on |+>':<28}")
for name, op in gates.items():
    a = zero.evolve(op).data
    b = plus.evolve(op).data
    print(f"{name:<5}{str(np.round(a,3)):<28}{str(np.round(b,3)):<28}")

# Known identities
S, T, Z, X, H = (gates[k].data for k in 'STZXH')
assert np.allclose(S @ S, Z)            # S^2 = Z
assert np.allclose(T @ T, S)            # T^2 = S
assert np.allclose(H @ H, np.eye(2))    # H^2 = I
assert np.allclose(X @ X, np.eye(2))    # X^2 = I
assert np.allclose(H @ Z @ H, X)        # HZH = X
assert np.allclose(H @ X @ H, Z)        # HXH = Z
print("Gate identities verified: S²=Z, T²=S, H²=X²=I, HZH=X, HXH=Z")

In [ ]:
# Same gates as circuits (gate-by-gate circuit builder)
def gate_circuit(sequence):
    qc = QuantumCircuit(1)
    for g in sequence:
        getattr(qc, g.lower())(0)
    return qc

qc = gate_circuit(['H', 'T', 'H', 'S', 'X'])
print(qc.draw())
final = Statevector.from_instruction(qc)
print("Final state:", np.round(final.data, 4))
# Circuit unitary equals the product of the matrices (later gates multiply on the left)
U_total = X @ S @ H @ T @ H
assert np.allclose(Operator(qc).data, U_total)

## 5. Bloch sphere visualization
For $|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$, the Bloch vector is
$(\sin\theta\cos\phi,\ \sin\theta\sin\phi,\ \cos\theta)$.

In [ ]:
def bloch_vector(sv):
    rho = np.outer(sv.data, sv.data.conj())
    x = 2*np.real(rho[0, 1])
    y = 2*np.imag(rho[1, 0])
    z = np.real(rho[0, 0] - rho[1, 1])
    return np.array([x, y, z])

expected = {
    '|0>': (zero, [0, 0, 1]), '|1>': (one, [0, 0, -1]),
    '|+>': (plus, [1, 0, 0]), '|->': (minus, [-1, 0, 0]),
    '|+i>': (plus_i, [0, 1, 0]), '|-i>': (minus_i, [0, -1, 0]),
}
for name, (sv, vec) in expected.items():
    bv = bloch_vector(sv)
    print(f"{name:5s} Bloch vector = {np.round(bv, 3)}")
    assert np.allclose(bv, vec)

In [ ]:
# Bloch sphere for the six cardinal states
fig_states = [('|0>', zero), ('|+>', plus), ('|+i>', plus_i)]
for name, sv in fig_states:
    display(plot_bloch_multivector(sv, title=name)) if 'display' in globals() else plot_bloch_multivector(sv, title=name)
plt.show()

## 6. Visualizing gate rotations
Each gate is a rotation of the Bloch sphere:
- **X, Y, Z**: π rotations about the x, y, z axes
- **S**: π/2 rotation about z; **T**: π/4 rotation about z
- **H**: π rotation about the (x+z)/√2 axis

In [ ]:
# Track the Bloch vector as a state moves through a gate sequence
def trajectory(sequence, start=zero):
    sv = start
    vecs = [bloch_vector(sv)]
    for g in sequence:
        sv = sv.evolve(gates[g])
        vecs.append(bloch_vector(sv))
    return np.array(vecs)

seq = ['H', 'T', 'T', 'S', 'H']
traj = trajectory(seq)
for i, v in enumerate(traj):
    label = 'start' if i == 0 else seq[i-1]
    print(f"after {label:>5}: {np.round(v, 3)}")

# Every Bloch vector stays on the unit sphere (unitary = rotation)
assert np.allclose(np.linalg.norm(traj, axis=1), 1.0)

In [ ]:
# Plot the path on a 3D sphere
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
ax.plot_wireframe(np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v), color='lightgray', linewidth=0.4)
ax.plot(*traj.T, 'o-', color='crimson')
for i, p in enumerate(traj):
    ax.text(*(p*1.1), 'start' if i == 0 else seq[i-1], fontsize=9)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title(f"Bloch trajectory: |0> -> {' -> '.join(seq)}")
plt.show()

In [ ]:
# Smooth rotation: sweep Rz and Ry to show continuous rotation about an axis
angles = np.linspace(0, 2*np.pi, 60)
start = plus  # on the equator
path_z = []
for a in angles:
    qc = QuantumCircuit(1); qc.rz(a, 0)
    path_z.append(bloch_vector(start.evolve(qc)))
path_z = np.array(path_z)

start_y = zero
path_y = []
for a in angles:
    qc = QuantumCircuit(1); qc.ry(a, 0)
    path_y.append(bloch_vector(start_y.evolve(qc)))
path_y = np.array(path_y)

fig = plt.figure(figsize=(11, 5))
for i, (path, title) in enumerate([(path_z, 'Rz sweep on |+> (equator)'), (path_y, 'Ry sweep on |0> (x-z plane)')]):
    ax = fig.add_subplot(1, 2, i+1, projection='3d')
    ax.plot_wireframe(np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v), color='lightgray', linewidth=0.3)
    ax.plot(*path.T, color='navy')
    ax.set_title(title)
plt.show()
assert np.allclose(np.linalg.norm(path_z, axis=1), 1) and np.allclose(np.linalg.norm(path_y, axis=1), 1)

## 7. Verifying each gate's rotation axis and angle
Check that each named gate matches the rotation $R_{\hat n}(\varphi) = e^{-i\varphi\,\hat n\cdot\vec\sigma/2}$ up to global phase.

In [ ]:
from scipy.linalg import expm
sx = gates['X'].data; sy = gates['Y'].data; sz = gates['Z'].data

def R(axis, phi):
    n = np.array(axis) / np.linalg.norm(axis)
    return expm(-1j * phi / 2 * (n[0]*sx + n[1]*sy + n[2]*sz))

def equal_up_to_phase(A, B):
    idx = np.unravel_index(np.argmax(np.abs(B)), B.shape)
    phase = A[idx] / B[idx]
    return np.allclose(A, phase * B)

table = {
    'X': ([1, 0, 0], np.pi),
    'Y': ([0, 1, 0], np.pi),
    'Z': ([0, 0, 1], np.pi),
    'S': ([0, 0, 1], np.pi/2),
    'T': ([0, 0, 1], np.pi/4),
    'H': ([1, 0, 1], np.pi),
}
for name, (axis, phi) in table.items():
    ok = equal_up_to_phase(gates[name].data, R(axis, phi))
    print(f"{name}: axis={axis}, angle={phi:.4f} rad -> match up to global phase: {ok}")
    assert ok

## 8. Mini-challenge: reach any point on the Bloch sphere
Using only $H$, $S$, $T$ (and their combinations), approximate a target state and report the fidelity.

In [ ]:
import itertools

target = Statevector([np.cos(0.6), np.exp(1j*1.1)*np.sin(0.6)])
alphabet = ['H', 'S', 'T']
best = (0, None)
for L in range(1, 9):
    for seq in itertools.product(alphabet, repeat=L):
        sv = zero
        for g in seq:
            sv = sv.evolve(gates[g])
        fid = abs(np.vdot(target.data, sv.data))**2
        if fid > best[0]:
            best = (fid, seq)
print(f"Best fidelity {best[0]:.4f} using sequence {' '.join(best[1])}")
assert best[0] > 0.9